# MindAI on Google Colab — Training + Persistent Brain Saves

This notebook runs `main_agent.py` directly on Google Colab and keeps the brain
persisted in Google Drive so you can resume training after Colab session limits.

## What it does

- mounts Google Drive;
- clones the repo and installs dependencies;
- syncs `savegame_brain/` from Drive into the Colab workspace before launch;
- runs `python main_agent.py` (optionally with `--c4`);
- `main_agent.py` autosaves the brain every **50,000 ticks**;
- after the run stops, the latest `savegame_brain/` is copied back to Google Drive.

## Important

`main_agent.py` already contains the needed logic:
- if `savegame_brain/` exists, it loads the latest brain state;
- during training it checkpoints every `50_000` ticks.

So on the next day you just run the notebook again, and training resumes from
that saved brain.


In [ ]:
# --- Cell 1: setup + Google Drive -------------------------------------------
import os, subprocess, shutil
from pathlib import Path

from google.colab import drive

drive.mount('/content/drive')

REPO = 'https://github.com/petryashov-a/MindAI.git'   # change to your fork if needed
if not os.path.exists('MindAI'):
    subprocess.run(['git', 'clone', '--depth=1', REPO], check=True)
%cd MindAI

!pip install -q numpy scipy torch tiktoken datasets av moviepy faster-whisper

DRIVE_ROOT = Path('/content/drive/MyDrive/mindai')
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
DRIVE_SAVE = DRIVE_ROOT / 'savegame_brain'

print('Drive save dir:', DRIVE_SAVE)


In [ ]:
# --- Cell 2: restore latest brain + optional data sync ----------------------
import shutil
from pathlib import Path

LOCAL_SAVE = Path('savegame_brain')
if DRIVE_SAVE.exists():
    if LOCAL_SAVE.exists():
        shutil.rmtree(LOCAL_SAVE)
    shutil.copytree(DRIVE_SAVE, LOCAL_SAVE)
    print('Restored brain from Google Drive ->', LOCAL_SAVE)
else:
    print('No previous brain save found in Google Drive; starting fresh.')

# Optional: if your dataset also lives in Google Drive, uncomment and adapt.
# DRIVE_DATA = DRIVE_ROOT / 'data'
# LOCAL_DATA = Path('data')
# if DRIVE_DATA.exists():
#     if LOCAL_DATA.exists():
#         shutil.rmtree(LOCAL_DATA)
#     shutil.copytree(DRIVE_DATA, LOCAL_DATA)
#     print('Restored data/ from Google Drive')


In [ ]:
# --- Cell 3: choose launch mode ---------------------------------------------
USE_C4 = True        # True -> adds --c4
DATA_DIR = 'data'    # default local data directory; change if needed

cmd = ['python', 'main_agent.py']
if USE_C4:
    cmd.append('--c4')
if DATA_DIR:
    cmd.extend(['--data', DATA_DIR])

print('Launch command:', ' '.join(cmd))


In [ ]:
# --- Cell 4: run training ---------------------------------------------------
# main_agent.py already:
#   - loads savegame_brain/ automatically if it exists
#   - saves every 50,000 ticks via checkpoint_interval=50000
# Stop the cell whenever you want; then run Cell 5 to copy the latest brain
# back to Google Drive.

import subprocess
subprocess.run(cmd, check=False)


## Save the latest brain back to Google Drive

After you stop Cell 4, run the next code cell to sync the latest
`savegame_brain/` checkpoint back to Google Drive.

Because `main_agent.py` saves every **50,000 ticks**, even if the Colab runtime
expires, you will usually keep the latest checkpoint rather than losing the
whole day.


In [ ]:
# --- Cell 5b: sync latest brain to Google Drive -----------------------------
import shutil
from pathlib import Path

LOCAL_SAVE = Path('savegame_brain')
if not LOCAL_SAVE.exists():
    print('No local savegame_brain/ found yet.')
else:
    if DRIVE_SAVE.exists():
        shutil.rmtree(DRIVE_SAVE)
    shutil.copytree(LOCAL_SAVE, DRIVE_SAVE)
    print('Synced latest brain to Google Drive ->', DRIVE_SAVE)
